<a href="https://colab.research.google.com/github/ks148678-beep/Data-Warehousing-and-Data-Mining/blob/main/Patient_clinic_data0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
# Load the dirty clinic patients dataset from the GitHub repository
url = "https://raw.githubusercontent.com/ks148678-beep/Data-Warehousing-and-Data-Mining/refs/heads/main/clinic_patients_dirty.csv"
df = pd.read_csv(url)
df.head()

,Patient ID,Full Name,SEX,Age (yrs),County,Visit_Date,Height,Weight (kg),BMI,Systolic BP,diastolic bp,Temp,Blood Group,Diagnosis,Smoker?
0,PT-0165,Purity Omondi,Female,19,Machakos,6/9/2025,164,83.8,21.7,142,?,36.9,O+,Hypertension,NaN
1,PT-0350,Halima Barasa,Female,45,Kiambu,01.06.2025,160,44.5,17.4,119,77,38.9,B+,Malaria,Yes
2,PT-0395,david otieno,Male,27,Nairobi,10/27/2025,177 cm,79.1,NaN,143,96,38.2,o+,Hypertension,No
3,PT-0075,Collins Kariuki,Male,28,Nairobi,5/29/2025,NaN,54.1,19.6,160,98,38,NaN,Hypertension,No
4,PT-0240,Grace Barasa,Female,25,Kiambu,3/13/2025,163,63.8 kg,24,121,71,37.9,O+,Typhoid,No


In [56]:
# Use Colab's built-in interactive table instead of Google Sheets
from google.colab import data_table
data_table.enable_dataframe_formatter()

# Display the DataFrame interactively
df

,Patient ID,Full Name,SEX,Age (yrs),County,Visit_Date,Height,Weight (kg),BMI,Systolic BP,diastolic bp,Temp,Blood Group,Diagnosis,Smoker?
0,PT-0165,Purity Omondi,Female,19,Machakos,6/9/2025,164,83.8,21.7,142,?,36.9,O+,Hypertension,NaN
1,PT-0350,Halima Barasa,Female,45,Kiambu,01.06.2025,160,44.5,17.4,119,77,38.9,B+,Malaria,Yes
2,PT-0395,david otieno,Male,27,Nairobi,10/27/2025,177 cm,79.1,NaN,143,96,38.2,o+,Hypertension,No
3,PT-0075,Collins Kariuki,Male,28,Nairobi,5/29/2025,NaN,54.1,19.6,160,98,38,NaN,Hypertension,No
4,PT-0240,Grace Barasa,Female,25,Kiambu,3/13/2025,163,63.8 kg,24,121,71,37.9,O+,Typhoid,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
415,PT-0338,Ruth Wambui,F,65,Nairobi,28-Oct-25,1.56m,80.1,32.9,141,86,36.7,B+,Type 2 Diabetes,TRUE
416,PT-0092,ruth kiplagat,Female,47,Kisumu,12/5/2025,158,62,30.8,?,75,39,O+,MALARIA,0
417,PT-0081,Brian Kamau,Male,36,Nakuru,7/20/2025,168,64.2,22.7,155,103,36.8°C,o+,Hypertension,0
418,PT-0192,Amina Barasa,Female,40,Mombassa,6/6/2025,157,66.6,27,126,83,37,AB+,Type 2 Diabetes,Yes


In [57]:
# Tidy column names (previously deleted cell step)
df.columns = (
    df.columns.str.strip()
   .str.lower()
   .str.replace(r"\(.*?\)", "", regex=True)
   .str.strip()
   .str.replace(r"[^a-z0-9]+", "_", regex=True)
   .str.strip("_")
)
df = df.rename(columns={
    "full_name": "patient_name",
    "age_yrs": "age",
    "visit_date": "visit_date",
    "weight_kg": "weight",
    "blood_group": "blood_group",
    "smoker": "smoker",
    "height": "height",
    "bmi": "bmi",
    "systolic_bp": "systolic_bp",
    "diastolic_bp": "diastolic_bp",
    "temp": "temp"
})

# Helper log function
log = []
def record(step, action, n):
    log.append({"step": step, "action": action, "affected": int(n)})
    print(f" -> {action}: {int(n)}")

# STEP 3: Strip whitespace + expose placeholders
before = df.copy()
df = df.apply(lambda s: s.str.strip() if s.dtype == object else s)
record(3, "whitespace trimmed", (before != df).sum().sum())

MISSING_TOKENS = {"", "n/a", "na", "null", "none", "nan", "?", "-", "unknown", "not recorded"}
is_missing = df.apply(lambda s: s.str.lower().isin(MISSING_TOKENS) if s.dtype==object else pd.Series(False, index=s.index))
print("\nPlaceholders:\n", df.where(is_missing).stack().value_counts().to_string())
df = df.mask(is_missing)
record(3, "placeholders -> NaN", is_missing.sum().sum())

print("\nMissing after exposing:\n", df.isna().sum().to_string())

 -> whitespace trimmed: 313

Placeholders:
 ?               21
-               19
not recorded    15
unknown         13
 -> placeholders -> NaN: 68

Missing after exposing:
 patient_id       0
patient_name     0
sex             17
age             22
county          21
visit_date      14
height          16
weight          22
bmi             23
systolic_bp     19
diastolic_bp    37
temp            17
blood_group     42
diagnosis       13
smoker          33


In [58]:

print("Duplicates before:", df.duplicated().sum())
df["patient_id"] = df["patient_id"].str.upper().str.replace(r"[\s_]+", "-", regex=True).str.strip()
n = len(df)
df = df.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)
record(4, "duplicate patient_id removed", n-len(df))

# STEP 5: Categorical cleaning
print("sex BEFORE:", sorted(df["sex"].dropna().unique()))
df["sex"] = df["sex"].str.lower().str.strip().map(
    {"m":"Male","male":"Male","f":"Female","female":"Female"}
)
print("sex AFTER:", sorted(df["sex"].dropna().unique()))

# County
city = df["county"].str.lower().str.replace(r"[^a-z ]","",regex=True).str.strip()
city = city.replace({"nairobbi":"nairobi","machackos":"machakos","nrb":"nairobi","mombassa":"mombasa"})
df["county"] = city.str.title()


bg = df["blood_group"].str.upper().str.replace(r"\s+","",regex=True)
bg = bg.replace({"OPOS":"O+","BPOS":"B+","APOS":"A+","ONEG":"O-","ANEG":"A-","BNEG":"B-","ABPOS":"AB+"})
df["blood_group"] = bg

diag = df["diagnosis"].str.lower().str.strip()
diag = diag.replace({
    "maleria":"malaria","high blood pressure":"hypertension",
    "type 2 diabetes":"diabetes","urti":"upper respiratory tract infection",
    "pneumonia":"pneumonia"
})
df["diagnosis"] = diag.str.title()


sm = df["smoker"].str.lower().map({"yes":"Yes","y":"Yes","true":"Yes","no":"No","n":"No","false":"No"})
df["smoker"] = sm


df["patient_name"] = df["patient_name"].str.title().str.replace(r"\s+"," ",regex=True)

Duplicates before: 12
 -> duplicate patient_id removed: 20
sex BEFORE: ['F', 'FEMALE', 'Female', 'M', 'MALE', 'Male', 'female', 'male']
sex AFTER: ['Female', 'Male']


In [59]:
# Height: "177 cm", "1.62" -> 162 cm
h = df["height"].astype(str)
# if value like 1.62 (meters) convert to cm
def parse_height(x):
    try:
        if pd.isna(x): return np.nan
        x=str(x).lower().replace("cm","").strip()
        v=float(x)
        if v < 3: # meters like 1.62
            v = v*100
        return v
    except: return np.nan

df["height"] = df["height"].apply(parse_height)

# Weight: "63.8 kg"
df["weight"] = pd.to_numeric(df["weight"].astype(str).str.replace(r"[^0-9.\-]","",regex=True), errors="coerce")

# Temp: handle F to C, e.g. 98.6F -> 37C
def parse_temp(x):
    if pd.isna(x): return np.nan
    s=str(x).lower()
    try:
        num = float(''.join(c for c in s if c.isdigit() or c=='.' or c=='-'))
        if 'f' in s: # Fahrenheit
            return round((num-32)*5/9,1)
        return num
    except: return np.nan
df["temp"] = df["temp"].apply(parse_temp)

# BP: "102/66" -> split, "-1" invalid
def parse_sys(x):
    if pd.isna(x): return np.nan
    s=str(x)
    if '/' in s:
        s=s.split('/')[0]
    try:
        v=float(s)
        return v if v>0 else np.nan
    except: return np.nan
df["systolic_bp"] = df["systolic_bp"].apply(parse_sys)
df["diastolic_bp"] = pd.to_numeric(df["diastolic_bp"], errors="coerce")
df["diastolic_bp"] = df["diastolic_bp"].apply(lambda v: v if pd.isna(v) or v>0 else np.nan)

df["age"] = pd.to_numeric(df["age"], errors="coerce").astype("Int64")
df["bmi"] = pd.to_numeric(df["bmi"], errors="coerce")


date_txt = df["visit_date"]
parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
for fmt in ["%d/%m/%Y","%m/%d/%Y","%d.%m.%Y","%d-%b-%y","%Y-%m-%d"]:
    attempt = pd.to_datetime(date_txt, format=fmt, errors="coerce")
    parsed = parsed.fillna(attempt)
df["visit_date"] = parsed
print(df.dtypes)

patient_id              object
patient_name            object
sex                     object
age                      Int64
county                  object
visit_date      datetime64[ns]
height                 float64
weight                 float64
bmi                    float64
systolic_bp            float64
diastolic_bp           float64
temp                   float64
blood_group             object
diagnosis               object
smoker                  object
dtype: object


In [60]:
df.loc[~df["age"].between(0,100), "age"] = pd.NA
record(7, "invalid age -> NaN", 0)
df.loc[~df["height"].between(100,220), "height"] = np.nan
df.loc[~df["weight"].between(20,200), "weight"] = np.nan
df.loc[~df["temp"].between(35,42), "temp"] = np.nan

# STEP 8 & 9: Repair BMI = weight / (height_m)^2
# Recalculate BMI where possible
height_m = df["height"]/100
expected_bmi = df["weight"] / (height_m**2)
# If BMI missing but we have height+weight, recover
mask = df["bmi"].isna() & expected_bmi.notna()
df.loc[mask, "bmi"] = expected_bmi[mask].round(1)
record(9, "BMI recovered from height/weight", mask.sum())

# STEP 10: Impute remaining
df["age"] = df["age"].fillna(int(df["age"].median()))
df["height"] = df["height"].fillna(df["height"].median())
df["weight"] = df["weight"].fillna(df["weight"].median())
df["bmi"] = df["bmi"].fillna(df["bmi"].median())
df["systolic_bp"] = df["systolic_bp"].fillna(df["systolic_bp"].median())
df["diastolic_bp"] = df["diastolic_bp"].fillna(df["diastolic_bp"].median())
df["temp"] = df["temp"].fillna(df["temp"].median())

for col in ["sex","county","blood_group","diagnosis","smoker"]:
    df[col] = df[col].fillna("Unknown")

# Total rebuilt
df["bmi"] = (df["weight"] / ((df["height"]/100)**2)).round(1)

# STEP 11: Export
df = df.sort_values("patient_id").reset_index(drop=True)
df.to_csv(OUT_PATH, index=False)
print(f"Clean saved to {OUT_PATH}, Rows: {len(raw)} -> {len(df)}")
print(pd.DataFrame(log).to_string(index=False))
df.head()

 -> invalid age -> NaN: 0
 -> BMI recovered from height/weight: 20
Clean saved to clinic_patients_clean.csv, Rows: 420 -> 400
 step                           action  affected
    3               whitespace trimmed       313
    3              placeholders -> NaN        68
    4     duplicate patient_id removed        20
    7               invalid age -> NaN         0
    9 BMI recovered from height/weight        20


,patient_id,patient_name,sex,age,county,visit_date,height,weight,bmi,systolic_bp,diastolic_bp,temp,blood_group,diagnosis,smoker
0,PT-0001,George Otieno,Male,70,Nairobi,1925-05-20,177.0,84.1,26.8,117.0,69.0,37.4,A-,Unknown,Yes
1,PT-0002,Ruth Kiplagat,Female,44,Kiambu,2025-02-10,159.0,76.3,30.2,119.0,71.0,36.6,A+,Diabetes,No
2,PT-0003,Lucy Kariuki,Female,55,Nakuru,2025-05-14,153.0,60.2,25.7,106.0,78.0,37.4,Unknown,Malaria,Yes
3,PT-0004,Samuel Abdi,Male,32,Nakuru,2025-07-06,176.0,88.3,28.5,123.0,79.0,37.4,A+,Upper Respiratory Tract Infection,No
4,PT-0005,Tabitha Kariuki,Female,78,Nairobi,2025-07-20,173.0,84.7,28.3,113.0,72.0,38.4,B+,Pneumonia,No
